# 05 · Predict

Scores unlabelled products with a bundle from notebook 04. The bundle carries the
cleaner, token vocabulary and vectorizer used in training, so new descriptions are
cleaned and featurised exactly as the training data was. Every input row comes back,
including rows the model cannot read, which are flagged rather than dropped.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # repo root, so `import classifier` works

DATA_DIR, CONFIG_DIR, OUTPUT_DIR = "../data", "../config", "../outputs"

import numpy as np
import pandas as pd

from classifier import io
from classifier.model import load_bundle, attach_names

STAMP = io.stamp()
print("run stamp:", STAMP)

## Parameters

In [ ]:
MODEL_DIR = "../outputs/model_YYYYMMDD_HHMMSS"   # from notebook 04

SOURCE = "file"                    # "file" | "sql"
INPUT_FILE = "to_predict.txt"      # in data/, pipe-delimited; used when SOURCE = "file"
CARRY_COLS = []                    # extra input columns to copy to the output
CHUNK_SIZE = 50000                 # rows per prediction chunk (memory)
ATTACH_NAMES = True                # add category names from the bundle's lookup

In [ ]:
# Products with no category yet. Returns the same columns as the training pull, so any
# marker the model was trained with can be attached.
SQL = """
select s.id as `ID`,
       s.erp_cono as `Cono`,
       s.erp_product_number as `Vallen ID`,
       replace(i.descrip, ';', ' ') as `FullDesc`,
       s.global_vendor_number as `Global Vendno`,
       i.prodcat as `ICSP Prodcat`,
       i.zzunspsc as `UNSPSC`,
       concat(s.erp_cono, w.prodline) as `Prodline`
from prd_bronze.stibo_ftp.sku s
inner join prd_bronze.openedge.icsp i
    on s.erp_product_number = i.prod
left join prd_bronze.openedge.icsw w
    on s.erp_product_number = w.prod
    and w.st_row_current = 1
    and w.cono = 1
    and w.whse = 'zdat'
where s.st_row_current = 1
    and s.category_l4_id is null
    and i.st_row_current = 1
    and i.cono in (1, 55)
    and i.descrip is not null
"""

## 1 · The model

In [ ]:
bundle = load_bundle(MODEL_DIR)
clf, cleaner, cleaning_config = bundle["clf"], bundle["cleaner"], bundle["cleaning_config"]
cfg = clf.cfg
TEXT_COL, ID_COL, ORIG_COL = cfg.text_col, cfg.id_col, cleaning_config["orig_col"]
print(f"model: {cfg.model_type}, trained {bundle['manifest']['stamp']} on "
      f"{bundle['manifest']['n_training_rows']:,} rows")
print(f"paths: {clf.hierarchy.n_paths:,} | features: {len(clf.vectorizer.vocabulary_):,} | "
      f"path mode: {cfg.path_mode}")
print(cleaner.describe())

## 2 · Input

In [ ]:
if SOURCE == "file":
    records = io.read_table(f"{DATA_DIR}/{INPUT_FILE}")
elif SOURCE == "sql":
    records = io.spark_to_pandas(spark.sql(SQL))       # noqa: F821
else:
    raise ValueError(f"SOURCE={SOURCE!r}")

needed = [ID_COL, TEXT_COL] + list(cleaner.markers) + CARRY_COLS
missing = [c for c in needed if c not in records.columns]
if missing:
    raise KeyError(f"input lacks {missing}. Marker columns are required because the model was "
                   f"trained with them: {list(cleaner.markers)}. Columns present: {list(records.columns)}")
print(f"{len(records):,} rows to predict")
records.head()

## 3 · Clean and predict

In [ ]:
cleaned = cleaner.clean_frame(records, TEXT_COL, orig_col=ORIG_COL, drop_empty=False)
pred = clf.predict(cleaned[TEXT_COL], chunk_size=CHUNK_SIZE)
if ATTACH_NAMES and bundle["lookup"] is not None:
    pred = attach_names(pred, bundle["lookup"], cfg.level_columns, cfg.name_columns)

out = pd.concat([cleaned[[ID_COL] + CARRY_COLS + [ORIG_COL, TEXT_COL]]
                 .rename(columns={TEXT_COL: "Cleaned Description"}), pred], axis=1)
assert len(out) == len(records), "rows lost"
out.head(15)

### What came back

`L<d> Confidence` is the share of path probability under the predicted prefix down to
depth d. The accuracy that goes with a confidence level can only be measured on labelled
data — read it off notebook 02 or 03's bucket tables and apply it to the volumes here.
Unreadable rows (`readable = False`) share no feature with the training data; their
prediction reflects class frequencies only.

In [ ]:
conf_cols = [f"L{d} Confidence" for d in range(1, len(cfg.level_columns) + 1)]
display(out[conf_cols].describe().T)

bands = [0, .5, .6, .7, .8, .9, .95, 1.0]
share = pd.DataFrame({c: pd.cut(out[c], bands, include_lowest=True).value_counts(normalize=True)
                      .sort_index() for c in conf_cols})
display(share.style.format("{:.1%}"))
print(f"unreadable: {int((~out['readable']).sum()):,} of {len(out):,}")

## 4 · Save

In [ ]:
OUT_FILE = f"{OUTPUT_DIR}/predictions_{STAMP}.txt"
io.write_table(out, OUT_FILE)
io.save_json({"stamp": STAMP, "model_dir": MODEL_DIR, "source": SOURCE,
              "input_file": INPUT_FILE if SOURCE == "file" else None, "n_rows": len(out)},
             f"{OUTPUT_DIR}/predictions_{STAMP}.json")